In [1]:
import sqlite3
import pandas as pd

# Load datasets
athletes = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/athlete_events.csv")
regions = pd.read_csv("https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-07-27/noc_regions.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")
sales = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv", sep='\t')


# Connect to SQLite in-memory DB
conn = sqlite3.connect(":memory:")

# Write DataFrames to SQL tables
athletes.to_sql("athletes_table", conn, index=False, if_exists="replace")
regions.to_sql("regions_table", conn, index=False, if_exists="replace")
sales.to_sql("sales_table", conn, index=False, if_exists="replace")

4622

In [2]:
##Count the total number of medals won by each country and show the top 5.
pd.read_sql(
    """
SELECT
    r.region AS country,
    COUNT(a.Medal) AS total_medals
FROM athletes_table AS a
JOIN regions_table AS r
    ON a.NOC = r.NOC
WHERE a.Medal IS NOT NULL
GROUP BY r.region
ORDER BY total_medals DESC
LIMIT 5;
""", conn
)

,country,total_medals
0,USA,5637
1,Russia,3947
2,Germany,3756
3,UK,2068
4,France,1777


In [3]:
##Calculate the average age of athletes who won a Gold medal.
pd.read_sql(
    """
SELECT
    AVG(Age) AS average_age_gold_medalists
FROM athletes_table
WHERE Medal = 'Gold'
    AND Age IS NOT NULL;
    """, conn
)

,average_age_gold_medalists
0,25.901013


In [4]:
##How many distinct events are there in each sport?
pd.read_sql(
    """
SELECT
    Sport,
    COUNT(DISTINCT Event) AS number_of_events
FROM athletes_table
GROUP BY Sport
ORDER BY number_of_events DESC;
""", conn
)

,Sport,number_of_events
0,Shooting,83
1,Athletics,83
2,Swimming,55
3,Cycling,44
4,Sailing,38
...,...,...
61,Cricket,1
62,Basque Pelota,1
63,Baseball,1
64,Alpinism,1


In [5]:
##Show all athletes from the United States (NOC = 'USA').
pd.read_sql(
    """
SELECT *
FROM athletes_table
WHERE NOC = 'USA';
""", conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None
1,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 50 kilometres,None
2,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 10/15 kilometres Pu...,None
3,6,Per Knut Aaland,M,31.0,188.0,75.0,United States,USA,1992 Winter,1992,Winter,Albertville,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None
4,6,Per Knut Aaland,M,33.0,188.0,75.0,United States,USA,1994 Winter,1994,Winter,Lillehammer,Cross Country Skiing,Cross Country Skiing Men's 10 kilometres,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18848,135458,Rami Zur,M,27.0,175.0,77.0,United States,USA,2004 Summer,2004,Summer,Athina,Canoeing,"Canoeing Men's Kayak Doubles, 500 metres",None
18849,135458,Rami Zur,M,31.0,175.0,77.0,United States,USA,2008 Summer,2008,Summer,Beijing,Canoeing,"Canoeing Men's Kayak Singles, 500 metres",None
18850,135458,Rami Zur,M,31.0,175.0,77.0,United States,USA,2008 Summer,2008,Summer,Beijing,Canoeing,"Canoeing Men's Kayak Singles, 1,000 metres",None
18851,135543,"Victor Andrew ""Vic"" Zwolak",M,25.0,175.0,64.0,United States,USA,1964 Summer,1964,Summer,Tokyo,Athletics,"Athletics Men's 3,000 metres Steeplechase",None


In [6]:
##Count how many medals were awarded each year.
pd.read_sql(
    """
SELECT
    Year,
    COUNT(Medal) AS total_medals
FROM athletes_table
WHERE Medal IS NOT NULL
    AND Medal <> 'NA'
GROUP BY Year
ORDER BY Year;
""", conn
)

,Year,total_medals
0,1896,143
1,1900,604
2,1904,486
3,1906,458
4,1908,831
5,1912,941
6,1920,1308
7,1924,962
8,1928,823
9,1932,739


In [7]:
##Find all athlete records where height or weight is missing.
pd.read_sql(
    """
SELECT *
FROM athletes_table
WHERE Height IS NULL
    OR Weight IS NULL;
""", conn
)


,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
1,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
2,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 100 metres,None
3,8,"Cornelia ""Cor"" Aalten (-Strannood)",F,18.0,168.0,NaN,Netherlands,NED,1932 Summer,1932,Summer,Los Angeles,Athletics,Athletics Women's 4 x 100 metres Relay,None
4,10,"Einar Ferdinand ""Einari"" Aalto",M,26.0,NaN,NaN,Finland,FIN,1952 Summer,1952,Summer,Helsinki,Swimming,Swimming Men's 400 metres Freestyle,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64258,135539,Marius Edmund Zwiller,M,18.0,NaN,NaN,France,FRA,1924 Summer,1924,Summer,Paris,Swimming,Swimming Men's 200 metres Breaststroke,None
64259,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 15 kilometres,None
64260,135542,Werner Zwingli,M,29.0,NaN,NaN,Switzerland,SUI,1956 Winter,1956,Winter,Cortina d'Ampezzo,Cross Country Skiing,Cross Country Skiing Men's 4 x 10 kilometres R...,None
64261,135552,Jan (Johann-) Zybert (Siebert-),M,20.0,NaN,NaN,Poland,POL,1928 Summer,1928,Summer,Amsterdam,Cycling,"Cycling Men's Team Pursuit, 4,000 metres",None


In [8]:
pd.read_sql(
    """
 SELECT AVG(Height) AS avg_height
FROM athletes_table;
""", conn
)

,avg_height
0,175.33897


In [9]:
##Replace the missing height with the average athlete height
conn.execute(
    """
UPDATE athletes_table
SET Height = (SELECT AVG(Height) FROM athletes_table)
WHERE Height IS NULL;
    """
)
conn.commit()



In [10]:
# Verify a few updated rows where Height was previously NULL
pd.read_sql(
    """
SELECT COUNT(*)
FROM athletes_table
WHERE Height IS NULL;
    """, conn
)


,COUNT(*)
0,0


In [11]:
pd.read_sql(
    """
SELECT *
FROM athletes_table
LIMIT 6;
    """, conn
)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.00000,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,None
1,2,A Lamusi,M,23.0,170.00000,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,None
2,3,Gunnar Nielsen Aaby,M,24.0,175.33897,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,None
3,4,Edgar Lindenau Aabye,M,34.0,175.33897,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.00000,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,None
5,5,Christine Jacoba Aaftink,F,21.0,185.00000,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,"Speed Skating Women's 1,000 metres",None


In [12]:
##Overwritting due to use of similar variable name - Sales
##Return total sales value per item.
pd.read_sql(
    """
SELECT
    item_name,
    SUM(
        quantity *
        CAST(REPLACE(item_price, '$', '') AS REAL)
    ) AS total_sales
FROM sales_table
GROUP BY item_name
    """, conn
)


,item_name,total_sales
0,6 Pack Soft Drink,369.93
1,Barbacoa Bowl,672.36
2,Barbacoa Burrito,894.75
3,Barbacoa Crispy Tacos,138.71
4,Barbacoa Salad Bowl,106.40
5,Barbacoa Soft Tacos,250.46
6,Bottled Water,649.18
7,Bowl,74.00
8,Burrito,44.40
9,Canned Soda,191.84


In [13]:
##Show the top 5 records with the highest item_price.
pd.read_sql(
    """
SELECT *
FROM sales_table
ORDER BY CAST(REPLACE(item_price, '$', '') AS REAL) DESC
LIMIT 5;
    """, conn
)

,order_id,quantity,item_name,choice_description,item_price
0,1443,15,Chips and Fresh Tomato Salsa,None,$44.25
1,1398,3,Carnitas Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",$35.25
2,511,4,Chicken Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$35.00
3,1443,4,Chicken Burrito,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",$35.00
4,1443,3,Veggie Burrito,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",$33.75


In [14]:
##How many unique customer orders are there? (Assume each order_id is a customer.)
pd.read_sql(
    """
SELECT
    COUNT(DISTINCT order_id) AS unique_orders
FROM sales_table;
    """, conn
)

,unique_orders
0,1834


In [16]:
##Find countries with high-performing athletes in the Olympics
pd.read_sql(
    """
SELECT
    country,
    medalists_count,
    avg_age,
    CASE
        WHEN medalists_count >= 1000 THEN 'High'
        WHEN medalists_count >= 500 THEN 'Medium'
        ELSE 'Low'
    END AS performance
FROM
(
    SELECT
        r.region AS country,
        COUNT(DISTINCT a.ID) AS medalists_count,
        AVG(a.Age) AS avg_age
    FROM athletes_table a
    JOIN regions_table r
        ON a.NOC = r.NOC
    WHERE a.Medal IS NOT NULL
    GROUP BY r.region
) AS country_stats
ORDER BY medalists_count DESC;
    """, conn
)

,country,medalists_count,avg_age,performance
0,USA,3836,24.896638,High
1,Russia,2612,25.286765,High
2,Germany,2565,25.464410,High
3,UK,1601,27.846310,High
4,France,1277,27.982153,High
...,...,...,...,...
131,Sudan,1,23.000000,Low
132,Suriname,1,22.000000,Low
133,Togo,1,27.000000,Low
134,Tonga,1,26.000000,Low
